**420-A58-SF - Algorithmes d'apprentissage non supervisé - Hiver 2026 - Spécialisation technique en Intelligence Artificielle**<br/>
MIT License - Copyright (c) 2026 Mikaël Swawola

# Examen #1 - Partitionnement de données en astrophysique - Solutions

| | |
|---|---|
| **Durée** | 3 heures 30 minutes |
| **Travail** | Individuel |
| **Documents** | Autorisés |
| **Remise** | Remettre ce notebook complété sur Lea |

## Contexte

L'astrophysique moderne génère des volumes massifs de données provenant de relevés astronomiques. Le **Sloan Digital Sky Survey (SDSS)** est l'un des plus grands relevés astronomiques, ayant catalogué des millions d'objets célestes avec des mesures photométriques dans cinq bandes spectrales (u, g, r, i, z). La classification automatique de ces objets (étoiles, galaxies, quasars) est un défi majeur où les techniques de partitionnement jouent un rôle central.

Dans cette évaluation, vous appliquerez différents algorithmes de partitionnement:
1. **Partie A** — Des données photométriques **inspirées** du SDSS pour classifier des objets célestes
2. **Partie B** — Une image du champ profond de Hubble pour détecter des galaxies avec DBSCAN

## Sources des données

**Données tabulaires:** Données photométriques générées à partir des distributions statistiques du *Sloan Digital Sky Survey Data Release 18* (SDSS DR18). Le SDSS est un projet ouvert dont les données sont librement accessibles : https://www.sdss.org/dr18/

**Image astronomique:** Image du *Hubble Deep Field* provenant de la librairie scikit-image, issue du télescope spatial Hubble (NASA/ESA). Images dans le domaine public selon la politique de la NASA : https://www.nasa.gov/nasa-brand-center/images-and-media/

## Barème

| Section | Question | Points |
|---------|----------|--------|
| **Partie A — Données tabulaires (55 points)** | | |
| 1 - Exploration des données stellaires | Q1.1 - Chargement et dimensions | 3 |
| | Q1.2 - Statistiques descriptives | 4 |
| | Q1.3 - Mise à l'échelle | 3 |
| | Q1.4 - Visualisation avec PCA | 5 |
| 2 - Partitionnement K-moyennes | Q2.1 - Standardisation des données | 5 |
| | Q2.2 - Application de K-moyennes (K=3) | 5 |
| | Q2.3 - Visualisation des clusters | 6 |
| 3 - Validation du partitionnement | Q3.1 - Score de Silhouette | 6 |
| | Q3.2 - Score de Calinski-Harabasz | 6 |
| | Q3.3 - K optimal et interprétation | 6 |
| | Q3.4 - Interprétation astrophysique | 6 |
| **Partie B — Détection de galaxies par image (45 points)** | | |
| 4 - Chargement et prétraitement | Q4.1 - Charger et visualiser l'image | 3 |
| | Q4.2 - Conversion en niveaux de gris | 3 |
| | Q4.3 - Seuillage de l'image | 4 |
| | Q4.4 - Extraction des coordonnées | 5 |
| 5 - Application de DBSCAN | Q5.1 - Appliquer DBSCAN | 6 |
| | Q5.2 - Nombre de galaxies détectées | 4 |
| | Q5.3 - Visualisation des clusters | 6 |
| | Q5.4 - Centroïdes des galaxies | 4 |
| 6 - Optimisation et analyse | Q6.1 - Variation des paramètres | 5 |
| | Q6.2 - Questions de réflexion | 5 |
| **Total** | | **100** |

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="darkgrid")
sns.set_context("notebook", font_scale=1.5, rc={"lines.linewidth": 2.5})
plt.rcParams['figure.figsize'] = (12, 8)

---
# Partie A — Partitionnement d'objets célestes à partir de données tabulaires (55 points)

Le fichier `sdss_stellar.csv` contient 800 observations d'objets célestes avec 8 variables :
- **u, g, r, i, z** : magnitudes photométriques dans les 5 filtres du SDSS (ultraviolet, vert, rouge, infrarouge proche)
- **redshift** : décalage vers le rouge spectroscopique
- **u_g, g_r** : indices de couleur (différences de magnitudes entre bandes adjacentes)

---
## 1 - Exploration des données stellaires (15 points)

**Q1.1 - Charger le fichier `sdss_stellar.csv` situé dans le dossier `data` à l'aide de Pandas. Afficher les dimensions du jeu de données (nombre d'observations et de variables).** (3 points)

In [ ]:
# Charger les données
df = pd.read_csv("../../data/sdss_stellar.csv")
X = df.values

print(f"Dimensions du jeu de données: {df.shape}")
print(f"Nombre d'observations: {df.shape[0]}")
print(f"Nombre de variables: {df.shape[1]}")
df.head()

**Réponse Q1.1:** Le jeu de données contient **800 observations** et **8 variables** (colonnes) : u, g, r, i, z (magnitudes photométriques), redshift (décalage vers le rouge), u_g et g_r (indices de couleur).

**Q1.2 - Afficher les statistiques descriptives des données (moyenne, écart-type, min, max, etc.). Commenter brièvement la distribution des variables.** (4 points)

In [ ]:
# Statistiques descriptives
df.describe()

**Réponse Q1.2:** Les magnitudes photométriques (u, g, r, i, z) ont des moyennes allant d'environ 17 à 20, avec des écarts-types autour de 1-1.6. La variable `redshift` a une moyenne d'environ 0.49 avec un écart-type de 0.86, ce qui montre une grande variabilité (présence d'objets proches et lointains). Les indices de couleur (u_g, g_r) ont des écarts-types plus faibles (~0.5-0.65). Les variables ne sont pas à la même échelle.

**Q1.3 - Les données sont-elles à la même échelle ? Justifier votre réponse en comparant les moyennes et écarts-types des variables. Une mise à l'échelle sera-t-elle nécessaire avant le partitionnement ?** (3 points)

**Réponse Q1.3:** Non, les données ne sont **pas à la même échelle**. Les magnitudes (u, g, r, i, z) ont des moyennes entre 15 et 20, tandis que le redshift varie entre 0 et 3.5 et les indices de couleur entre -0.5 et 2.7. Les écarts-types diffèrent significativement (de 0.5 pour g_r à 1.6 pour u). Une **standardisation est nécessaire** avant d'appliquer K-moyennes, car cet algorithme utilise la distance euclidienne et serait dominé par les variables de plus grande amplitude.

**Q1.4 - Réaliser une réduction de dimensionnalité avec PCA (2 composantes) et visualiser les données projetées dans un diagramme en nuage de points. Combien de groupes pouvez-vous estimer visuellement ?** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>PCA(n_components=2)</code> de scikit-learn et <code>sns.scatterplot()</code> pour la visualisation</p>
</details>

In [ ]:
from sklearn.decomposition import PCA

# Réduction de dimensionnalité avec PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Visualisation
plt.figure(figsize=(12, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], alpha=0.6)
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title('Projection PCA des données stellaires SDSS')
plt.show()

print(f"Variance expliquée: {pca.explained_variance_ratio_}")
print(f"Variance totale expliquée: {pca.explained_variance_ratio_.sum():.2%}")

**Réponse Q1.4:** En utilisant la projection PCA en 2D, on peut observer visuellement environ **3 groupes** distincts. Un groupe est clairement séparé des deux autres (probablement les quasars, caractérisés par des redshifts élevés). Les deux autres groupes (étoiles et galaxies) sont plus proches mais tout de même distinguables.

---
## 2 - Partitionnement K-moyennes (16 points)

**Q2.1 - Standardiser les données à l'aide de `StandardScaler` de scikit-learn.** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>StandardScaler()</code> puis <code>fit_transform()</code> sur la matrice de données</p>
</details>

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Moyennes après standardisation: {X_scaled.mean(axis=0).round(2)}")
print(f"Écarts-types après standardisation: {X_scaled.std(axis=0).round(2)}")

**Q2.2 - Appliquer l'algorithme K-moyennes avec K=3 clusters sur les données standardisées. Afficher l'inertie (la distorsion) et le nombre d'itérations.** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>KMeans(n_clusters=3, n_init=10, random_state=42)</code> de scikit-learn</p>
</details>

In [ ]:
from sklearn.cluster import KMeans

# Application de K-moyennes avec K=3
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans.fit(X_scaled)

print(f"Inertie: {kmeans.inertia_:.2f}")
print(f"Nombre d'itérations: {kmeans.n_iter_}")

**Q2.3 - Visualiser les clusters obtenus (K=3) à l'aide d'un diagramme en nuage de points coloré par cluster. Utiliser les composantes principales (PCA) pour la projection en 2D.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Appliquer PCA sur les données standardisées puis utiliser <code>sns.scatterplot()</code> avec <code>hue=kmeans.labels_</code></p>
</details>

In [ ]:
# PCA sur les données standardisées
pca_scaled = PCA(n_components=2)
X_pca_scaled = pca_scaled.fit_transform(X_scaled)

# Visualisation des clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x=X_pca_scaled[:, 0], y=X_pca_scaled[:, 1], hue=kmeans.labels_, palette="Set2", alpha=0.7)
plt.xlabel('Composante principale 1')
plt.ylabel('Composante principale 2')
plt.title('Clusters K-moyennes (K=3) - Projection PCA des données stellaires')
plt.legend(title='Cluster')
plt.show()

---
## 3 - Validation du partitionnement (24 points)

**Q3.1 - Calculer le score de Silhouette pour K variant de 2 à 8. Afficher les résultats sous forme de graphique.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>silhouette_score(X_scaled, labels)</code> de scikit-learn pour chaque valeur de K</p>
</details>

In [ ]:
from sklearn.metrics import silhouette_score

# Calcul du score de Silhouette pour différentes valeurs de K
K_range = range(2, 9)
silhouette_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette = {score:.4f}")

# Graphique
plt.figure(figsize=(10, 6))
plt.plot(K_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Score de Silhouette')
plt.title('Score de Silhouette en fonction de K')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

**Q3.2 - Calculer le score de Calinski-Harabasz pour K variant de 2 à 8. Afficher les résultats sous forme de graphique.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>calinski_harabasz_score(X_scaled, labels)</code> de scikit-learn. Le score optimal est le <b>maximum</b>.</p>
</details>

In [ ]:
from sklearn.metrics import calinski_harabasz_score

# Calcul du score de Calinski-Harabasz pour différentes valeurs de K
calinski_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = calinski_harabasz_score(X_scaled, labels)
    calinski_scores.append(score)
    print(f"K={k}: Calinski-Harabasz = {score:.1f}")

# Graphique
plt.figure(figsize=(10, 6))
plt.plot(K_range, calinski_scores, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Nombre de clusters (K)')
plt.ylabel('Score de Calinski-Harabasz')
plt.title('Score de Calinski-Harabasz en fonction de K')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

**Q3.3 - D'après les scores de Silhouette et Calinski-Harabasz, quelle est la valeur optimale de K ? Les deux métriques convergent-elles vers la même valeur ?** (6 points)

**Réponse Q3.3:** Les deux métriques convergent vers **K=3** comme valeur optimale. Le score de Silhouette atteint son maximum à K=3 (~0.53), indiquant des clusters bien séparés et cohérents. Le score de Calinski-Harabasz est également maximal à K=3 (~1089), confirmant que la variance inter-clusters est maximisée par rapport à la variance intra-clusters. La convergence des deux métriques renforce la confiance dans le choix de K=3.

**Q3.4 - En astrophysique, les objets célestes se classifient naturellement en trois catégories : étoiles, galaxies et quasars. Le partitionnement obtenu est-il cohérent avec cette classification ? Quelles variables semblent les plus discriminantes pour séparer ces trois types d'objets ?** (6 points)

*Indice : analyser les centroïdes des clusters pour identifier les caractéristiques de chaque groupe.*

In [ ]:
# Analyser les centroïdes
kmeans_3 = KMeans(n_clusters=3, n_init=10, random_state=42)
labels_3 = kmeans_3.fit_predict(X_scaled)

# Centroïdes dans l'espace original
centroids_original = scaler.inverse_transform(kmeans_3.cluster_centers_)
centroids_df = pd.DataFrame(centroids_original, columns=df.columns)
centroids_df.index = [f'Cluster {i}' for i in range(3)]
print("Centroïdes des clusters (valeurs originales):")
centroids_df.round(3)

**Réponse Q3.4:** Oui, le partitionnement K=3 est cohérent avec la classification astrophysique naturelle :

- **Cluster « Étoiles »** : magnitudes plus faibles (objets plus brillants), redshift quasi nul (~0), indice u-g élevé (~1.5)
- **Cluster « Galaxies »** : magnitudes intermédiaires à élevées, redshift modéré (0.02-0.4), indice g-r élevé (~1.3)
- **Cluster « Quasars »** : magnitudes intermédiaires, redshift élevé (0.8-3.5), indice u-g faible (~0.2)

Les variables les plus discriminantes sont le **redshift** (qui sépare très bien les quasars des autres objets) et les **indices de couleur** u_g et g_r (qui distinguent les étoiles des galaxies).

---
# Partie B — Détection de galaxies dans une image avec DBSCAN (45 points)

Dans cette partie, nous utilisons l'image du **Hubble Deep Field** (champ profond de Hubble), l'une des images les plus célèbres en astronomie, capturée par le télescope spatial Hubble (NASA/ESA). Cette image révèle des milliers de galaxies lointaines. L'objectif est d'utiliser **DBSCAN** pour détecter et localiser automatiquement les objets lumineux (galaxies) dans cette image.

---
## 4 - Chargement et prétraitement de l'image (15 points)

**Q4.1 - Charger l'image du Hubble Deep Field depuis scikit-image (`data.hubble_deep_field()`) et la visualiser. Afficher ses dimensions.** (3 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>data.hubble_deep_field()</code> de scikit-image et <code>plt.imshow()</code></p>
</details>

In [ ]:
from skimage import data

# Charger l'image du Hubble Deep Field
image = data.hubble_deep_field()

# Visualiser l'image
plt.figure(figsize=(12, 10))
plt.imshow(image)
plt.title('Hubble Deep Field (NASA/ESA)')
plt.axis('off')
plt.show()

print(f"Dimensions de l'image: {image.shape}")
print(f"Type: {image.dtype}")

**Q4.2 - L'image est en couleur (RGB). Convertir l'image en niveaux de gris à l'aide de `rgb2gray` de scikit-image. Afficher l'image en niveaux de gris et ses valeurs min/max.** (3 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>rgb2gray(image)</code> de <code>skimage.color</code> et <code>plt.imshow(gray, cmap='gray')</code></p>
</details>

In [ ]:
from skimage.color import rgb2gray

# Conversion en niveaux de gris
gray = rgb2gray(image)

# Visualiser
plt.figure(figsize=(12, 10))
plt.imshow(gray, cmap='gray')
plt.title('Hubble Deep Field - Niveaux de gris')
plt.colorbar(label='Intensité')
plt.show()

print(f"Dimensions: {gray.shape}")
print(f"Valeur minimum: {gray.min():.4f}")
print(f"Valeur maximum: {gray.max():.4f}")

**Q4.3 - Appliquer un seuillage en utilisant la méthode d'Otsu (`threshold_otsu`) pour séparer les objets lumineux (galaxies) du fond sombre. Créer une image binaire et la visualiser.** (4 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>threshold_otsu(gray)</code> puis créer l'image binaire avec <code>binary = gray > threshold</code></p>
</details>

In [ ]:
from skimage.filters import threshold_otsu

# Calculer le seuil d'Otsu
threshold = threshold_otsu(gray)
print(f"Seuil d'Otsu calculé: {threshold:.4f}")

# Créer l'image binaire
binary = gray > threshold

# Visualiser
plt.figure(figsize=(12, 10))
plt.imshow(binary, cmap='gray')
plt.title('Image binaire après seuillage d\'Otsu')
plt.axis('off')
plt.show()

**Q4.4 - Extraire les coordonnées (y, x) de tous les pixels blancs (valeur True) de l'image binaire. Combien de pixels correspondent aux objets d'intérêt ?** (5 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>np.argwhere(binary)</code> pour obtenir les indices des pixels non nuls</p>
</details>

In [ ]:
# Extraire les coordonnées des pixels lumineux
coords = np.argwhere(binary)
print(f"Forme des coordonnées: {coords.shape}")
print(f"Nombre de pixels d'intérêt: {len(coords)}")

**Réponse Q4.4:** Environ **27 374 pixels** correspondent aux objets lumineux (galaxies et autres sources) dans l'image, ce qui représente environ 3.1% des pixels totaux (872 × 1000 = 872 000).

---
## 5 - Application de DBSCAN (20 points)

**Q5.1 - Instancier et entraîner un modèle DBSCAN sur les coordonnées des pixels lumineux. Utiliser `eps=10` et `min_samples=50` comme paramètres initiaux.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Utiliser <code>DBSCAN(eps=10, min_samples=50)</code> de scikit-learn et la méthode <code>fit()</code></p>
</details>

In [ ]:
from sklearn.cluster import DBSCAN

# Appliquer DBSCAN
dbscan = DBSCAN(eps=10, min_samples=50)
dbscan.fit(coords)

**Q5.2 - Récupérer les labels des clusters. Combien de galaxies (clusters) ont été détectées ? Combien de points sont classés comme bruit (label = -1) ?** (4 points)

In [ ]:
# Récupérer les labels
labels = dbscan.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = np.sum(labels == -1)

print(f"Nombre de clusters (galaxies) détectés: {n_clusters}")
print(f"Nombre de points de bruit: {n_noise}")
print(f"Pourcentage de bruit: {n_noise / len(labels) * 100:.1f}%")

**Réponse Q5.2:** Avec eps=10 et min_samples=50, DBSCAN détecte environ **92 clusters** (objets/galaxies) et classe environ **6 100 points** comme bruit (pixels isolés ne formant pas de cluster dense), soit environ 22% des pixels lumineux.

**Q5.3 - Créer une image des clusters en colorant chaque pixel selon son appartenance. Visualiser le résultat.** (6 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Créer une image vide de même taille, puis assigner à chaque pixel son label de cluster. Utiliser <code>cmap='nipy_spectral'</code></p>
</details>

In [ ]:
# Créer une image des clusters
cluster_image = np.zeros(gray.shape, dtype=int)
for idx, (y, x) in enumerate(coords):
    cluster_image[y, x] = labels[idx] + 1  # +1 pour décaler le bruit (-1) vers 0

plt.figure(figsize=(12, 10))
plt.imshow(cluster_image, cmap='nipy_spectral')
plt.title(f'Clusters détectés par DBSCAN (n={n_clusters})')
plt.colorbar(label='Cluster')
plt.axis('off')
plt.show()

**Q5.4 - Calculer les centroïdes (positions moyennes) de chaque cluster et les afficher superposés sur l'image originale.** (4 points)

<details>
<summary><font size="3" color="darkgreen"><b>Cliquer ici pour obtenir un indice</b></font></summary>
<p>Pour chaque label de cluster (différent de -1), calculer la moyenne des coordonnées y et x des pixels appartenant à ce cluster</p>
</details>

In [ ]:
# Calculer les centroïdes
centroids = []
for label in range(n_clusters):
    mask = labels == label
    cluster_coords = coords[mask]
    centroid = cluster_coords.mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print(f"Nombre de centroïdes calculés: {len(centroids)}")

# Visualiser l'image avec les centroïdes
plt.figure(figsize=(14, 12))
plt.imshow(image)
plt.scatter(centroids[:, 1], centroids[:, 0], c='red', s=80, marker='x', linewidths=2)
plt.title(f'Galaxies détectées par DBSCAN - {n_clusters} objets')
plt.axis('off')
plt.show()

---
## 6 - Optimisation et analyse (10 points)

**Q6.1 - Expérimenter avec différentes valeurs de `eps` (5, 10, 15, 20) et `min_samples` (30, 50, 80, 100). Observer l'impact sur le nombre de clusters détectés. Présenter les résultats dans une forme structurée (tableau ou affichage).** (5 points)

In [ ]:
# Tester différentes combinaisons de paramètres
eps_values = [5, 10, 15, 20]
min_samples_values = [30, 50, 80, 100]

print(f"{'eps':>5} | {'min_samples':>12} | {'Clusters':>10} | {'Bruit':>8}")
print("-" * 45)

for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms)
        db.fit(coords)
        n = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
        noise = np.sum(db.labels_ == -1)
        print(f"{eps:>5} | {ms:>12} | {n:>10} | {noise:>8}")

**Q6.2 - Répondre aux questions suivantes :** (5 points)

a) Pourquoi DBSCAN est-il particulièrement adapté à la détection de galaxies dans une image, comparé à K-moyennes ?

b) Quel est l'effet d'augmenter `eps` sur le nombre de clusters détectés ? Et l'effet d'augmenter `min_samples` ?

c) Quelles seraient les limites de cette approche pour détecter des galaxies dans des images astronomiques réelles de haute résolution ?

**Réponse Q6.2:**

a) DBSCAN est particulièrement adapté car :
- Il **n'exige pas de connaître le nombre de galaxies** à l'avance (contrairement à K-moyennes)
- Il **gère le bruit** automatiquement : les pixels isolés ou de faible intensité sont classés comme bruit et ne faussent pas les clusters
- Il peut détecter des **clusters de forme arbitraire** (les galaxies n'ont pas toutes la même forme : spirales, elliptiques, irrégulières)
- Les galaxies forment des **régions denses de pixels lumineux**, exactement le type de structure que DBSCAN est conçu pour identifier

b) **Augmenter `eps`** : des clusters proches fusionnent → le nombre de clusters **diminue**. Avec un eps trop grand, des galaxies distinctes mais proches seront regroupées en un seul cluster.
**Augmenter `min_samples`** : seuls les clusters suffisamment denses sont conservés → le nombre de clusters **diminue**. Les petites galaxies (peu de pixels lumineux) sont classées comme bruit.

c) Limites de cette approche :
- **Passage à l'échelle** : DBSCAN a une complexité O(n²) en mémoire, ce qui pose problème pour des images de très haute résolution (des millions de pixels)
- **Galaxies superposées** : si des galaxies se chevauchent (projection), DBSCAN peut les fusionner en un seul cluster
- **Sensibilité au seuil** : la qualité de la détection dépend fortement du seuillage initial
- **Variations de densité** : les galaxies ont des luminosités très différentes; un seul jeu de paramètres eps/min_samples ne convient pas à toutes
- **Artefacts** : les rayons cosmiques, étoiles de premier plan et reflets optiques peuvent créer de faux clusters

---
**Fin de l'examen #1**